# **Analisi esplorativa di `quotes.json`**



In [3]:
import pandas as pd

df_quotes = pd.read_json("quotes.json")

print("Dimensioni del dataset:", df_quotes.shape)
display(df_quotes.head())

Dimensioni del dataset: (10508, 5)


,timestamp,quoted_user,quoting_user,quoting_truth,quoted_truth_external_id
0,2022-07-22 09:43:00,125,131,16333,108688461672926240
1,2022-08-20 18:00:00,300,300,14949,108857444588213248
2,2022-07-14 21:13:00,2106,8212,367606,108648254976307344
3,2022-09-12 22:40:00,933,27,1593,108790130246424352
4,2022-11-11 00:00:00,134003,6657,763581,109324987069121056


In [5]:
df_quotes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10508 entries, 0 to 10507
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   timestamp                 10508 non-null  datetime64[ns]
 1   quoted_user               10508 non-null  int64         
 2   quoting_user              10508 non-null  int64         
 3   quoting_truth             10508 non-null  int64         
 4   quoted_truth_external_id  10508 non-null  int64         
dtypes: datetime64[ns](1), int64(4)
memory usage: 492.6 KB


In [7]:
#controlliamo se compaiono valori speciali come -1, visto che in users.xml era molto importante.
for column in ["quoted_user", "quoting_user", "quoting_truth", "quoted_truth_external_id"]:
    print(column, ":", (df_quotes[column] == -1).sum())

quoted_user : 0
quoting_user : 0
quoting_truth : 0
quoted_truth_external_id : 0


In [9]:
print("Data minima:", df_quotes["timestamp"].min())
print("Data massima:", df_quotes["timestamp"].max())

Data minima: 2022-06-08 13:39:00
Data massima: 2022-12-06 00:00:00


### Analisi degli utenti coinvolti nelle quote



In [11]:
print("Utenti citati distinti:", df_quotes["quoted_user"].nunique())
print("Utenti che citano distinti:", df_quotes["quoting_user"].nunique())

Utenti citati distinti: 2873
Utenti che citano distinti: 409


In [13]:
#utenti più citati
most_quoted_users = df_quotes["quoted_user"].value_counts().head(10)

display(most_quoted_users)

quoted_user
27       811
143      178
125      163
16675    146
416      130
1118     117
9365     104
974      103
322      100
178       97
Name: count, dtype: int64

E qui possiamo sfruttare users.xml: sappiamo già che id_user identifica univocamente gli utenti. Potremmo quindi scoprire chi è l’utente 27 facendo un collegamento tra quotes.json e users.xml.

In [16]:
#chi cita di più
most_quoting_users = df_quotes["quoting_user"].value_counts().head(10)

display(most_quoting_users)

quoting_user
4273     939
9365     734
10825    630
10804    337
2155     322
7630     243
4248     234
805      213
7585     203
12888    203
Name: count, dtype: int64

### Analisi degli identificativi dei post



In [19]:
print("Quoting truth distinti:", df_quotes["quoting_truth"].nunique())
print("Quoting truth duplicati:", df_quotes["quoting_truth"].duplicated().sum())

Quoting truth distinti: 10508
Quoting truth duplicati: 0


In [21]:
print("Quoted truth external ID distinti:",
      df_quotes["quoted_truth_external_id"].nunique())

print("Quoted truth external ID duplicati:",
      df_quotes["quoted_truth_external_id"].duplicated().sum())

Quoted truth external ID distinti: 9760
Quoted truth external ID duplicati: 748


### Analisi dei record duplicati


In [27]:
duplicated_rows = df_quotes.duplicated().sum()

print("Record completamente duplicati:", duplicated_rows)

Record completamente duplicati: 0


***verificare la coerenza logica tra quoted_user e quoting_user***

In [30]:
self_quotes = (df_quotes["quoted_user"] == df_quotes["quoting_user"]).sum()

print("Quote in cui l'utente cita sé stesso:", self_quotes)

Quote in cui l'utente cita sé stesso: 937


### Analisi della distribuzione temporale delle quote


In [34]:
print("Timestamp distinti:", df_quotes["timestamp"].nunique())
print("Timestamp ripetuti:", df_quotes["timestamp"].duplicated().sum())

Timestamp distinti: 8528
Timestamp ripetuti: 1980


### Coerenza tra post citato e autore

`quoted_truth_external_id` risulta associato sempre allo stesso `quoted_user`?

In [41]:
authors_per_quoted_truth = (
    df_quotes.groupby("quoted_truth_external_id")["quoted_user"]
    .nunique()
)

inconsistent_quoted_truths = authors_per_quoted_truth[
    authors_per_quoted_truth no > 1
]

print("Post citati associati a più utenti:",
      len(inconsistent_quoted_truths))

Post citati associati a più utenti: 0


collegare `quotes.json` con `users.xml` e `truths.csv` per verificare:

-  `quoted_user` e `quoting_user` corrispondono a utenti effettivamente presenti in `users.xml`?
-  `quoting_truth` identifica un post presente in `truths.csv` e  l'autore del post coincide con `quoting_user`?
- `quoted_truth_external_id` permette di identificare il post originale citato all'interno di `truths.csv`?
- l'autore del post originale individuato in `truths.csv` coincide con `quoted_user`?
